In [16]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import statsmodels.api as sm

In [25]:
df=pd.read_csv("../data/startups.csv")

In [3]:
df.head()

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94


In [4]:
df.describe(include='all')

,R&D Spend,Administration,Marketing Spend,State,Profit
count,50.000000,50.000000,50.000000,50,50.000000
unique,NaN,NaN,NaN,3,NaN
top,NaN,NaN,NaN,New York,NaN
freq,NaN,NaN,NaN,17,NaN
mean,73721.615600,121344.639600,211025.097800,NaN,112012.639200
std,45902.256482,28017.802755,122290.310726,NaN,40306.180338
min,0.000000,51283.140000,0.000000,NaN,14681.400000
25%,39936.370000,103730.875000,129300.132500,NaN,90138.902500
50%,73051.080000,122699.795000,212716.240000,NaN,107978.190000
75%,101602.800000,144842.180000,299469.085000,NaN,139765.977500


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [26]:
df = pd.get_dummies(df, columns=['State'], drop_first=True)
df.head()

,R&D Spend,Administration,Marketing Spend,Profit,State_Florida,State_New York
0,165349.20,136897.80,471784.10,192261.83,False,True
1,162597.70,151377.59,443898.53,191792.06,False,False
2,153441.51,101145.55,407934.54,191050.39,True,False
3,144372.41,118671.85,383199.62,182901.99,False,True
4,142107.34,91391.77,366168.42,166187.94,True,False


In [27]:

targets = df['Profit']
inputs = df.drop(['Profit'],axis=1)

# Spliting the variables with an 80-20 split and some random state
# random state(with any integer), makes the split same everytime we run the code. without it, we get different 
# splits every time we run the code. so You are not removing randomness — you are controlling it
x_train, x_test, y_train, y_test = train_test_split(inputs, targets, test_size=0.2, random_state=365)

In [28]:
inputs.head()

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
0,165349.20,136897.80,471784.10,False,True
1,162597.70,151377.59,443898.53,False,False
2,153441.51,101145.55,407934.54,True,False
3,144372.41,118671.85,383199.62,False,True
4,142107.34,91391.77,366168.42,True,False


In [11]:
targets.head()

0    192261.83
1    191792.06
2    191050.39
3    182901.99
4    166187.94
Name: Profit, dtype: float64

In [29]:
# Create a linear regression object
reg = LinearRegression()
# Fit the regression with the scaled TRAIN inputs and targets
reg.fit(x_train,y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [30]:
score =reg.score(x_train,y_train)
intercept =reg.intercept_
coefficents =reg.coef_
print("Score: ",score)
print("Intercept: ",intercept)
print("Coefficents: ",coefficents)

# Score is the r squared
# intercept is the y value when all x is 0
# since we scaled the data, “How much y changes for a 1 standard deviation increase in the feature”

Score:  0.9415157965234591
Intercept:  44798.56940925348
Coefficents:  [ 8.07001826e-01  1.43179967e-02  2.45926342e-02 -4.00710920e+01
  1.17698832e+02]


In [17]:
def backward_elimination(X, y, significance_level=0.05):
    X = X.copy()
    X = sm.add_constant(X)

    while True:
        model = sm.OLS(y, X).fit()
        p_values = model.pvalues.drop("const", errors="ignore")
        
        if p_values.empty:
            break

        max_p_value = p_values.max()
        
        if max_p_value > significance_level:
            feature_to_remove = p_values.idxmax()
            print(f"Removing {feature_to_remove} with p-value {max_p_value:.4f}")
            X = X.drop(columns=[feature_to_remove])
        else:
            break

    return X, model

In [31]:
inputs = inputs.replace({True: 1, False: 0})

/var/folders/qj/csh4ys_506qc4hr_myvhtfym0000gn/T/ipykernel_71131/4266430563.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  inputs = inputs.replace({True: 1, False: 0})


In [32]:
inputs.head()

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
0,165349.20,136897.80,471784.10,0,1
1,162597.70,151377.59,443898.53,0,0
2,153441.51,101145.55,407934.54,1,0
3,144372.41,118671.85,383199.62,0,1
4,142107.34,91391.77,366168.42,1,0


In [33]:
X_selected, final_model = backward_elimination(inputs,targets)

print("\nSelected features:")
print(X_selected.columns.tolist())

print("\nFinal model summary:")
print(final_model.summary())

Removing State_New York with p-value 0.9898
Removing State_Florida with p-value 0.9398
Removing Administration with p-value 0.6018
Removing Marketing Spend with p-value 0.0600

Selected features:
['const', 'R&D Spend']

Final model summary:
                            OLS Regression Results                            
Dep. Variable:                 Profit   R-squared:                       0.947
Model:                            OLS   Adj. R-squared:                  0.945
Method:                 Least Squares   F-statistic:                     849.8
Date:                Thu, 09 Apr 2026   Prob (F-statistic):           3.50e-32
Time:                        22:34:10   Log-Likelihood:                -527.44
No. Observations:                  50   AIC:                             1059.
Df Residuals:                      48   BIC:                             1063.
Df Model:                           1                                         
Covariance Type:            nonrobust           


  <h1>Backward Elimination in Python for Linear Regression</h1>

  <h2>What is Backward Elimination?</h2>
  <p>
    Backward elimination is a feature selection method used in linear regression.
    It starts with <b>all input features</b>, then removes the least useful feature
    one by one until only statistically significant features remain.
  </p>

  <div class="box">
    <b>Steps:</b>
    <ol>
      <li>Start with all features</li>
      <li>Fit the regression model</li>
      <li>Look at the p-value of each feature</li>
      <li>Remove the feature with the highest p-value if it is above the significance level</li>
      <li>Repeat until all remaining features are significant</li>
    </ol>
  </div>

  <h2>Python Code</h2>
  <pre><code>def backward_elimination(X, y, significance_level=0.05):
    X = X.copy()
    X = sm.add_constant(X)

    while True:
        model = sm.OLS(y, X).fit()
        p_values = model.pvalues.drop("const", errors="ignore")
        
        if p_values.empty:
            break

        max_p_value = p_values.max()
        
        if max_p_value > significance_level:
            feature_to_remove = p_values.idxmax()
            print(f"Removing {feature_to_remove} with p-value {max_p_value:.4f}")
            X = X.drop(columns=[feature_to_remove])
        else:
            break

    return X, model</code></pre>

  <h2>Code Explanation</h2>

  <h3>1. Function Definition</h3>
  <pre><code>def backward_elimination(X, y, significance_level=0.05):</code></pre>
  <p>
    <b>X</b> is the input features, <b>y</b> is the target variable, and
    <b>significance_level</b> is the threshold used to decide whether a feature
    should stay in the model or be removed.
  </p>

  <h3>2. Copy the Data and Add the Constant</h3>
  <pre><code>X = X.copy()
X = sm.add_constant(X)</code></pre>

  <p>
    <b>X.copy()</b> makes a copy so the original data is not changed.
  </p>

  <p>
    <b>sm.add_constant(X)</b> adds a new column called <span class="highlight">const</span>,
    which contains only 1s.
  </p>

  <h3>What is the Constant?</h3>
  <p>
    The constant is also called the <b>intercept</b>. A linear regression model looks like this:
  </p>

  <div class="box">
    <b>y = β₀ + β₁x₁ + β₂x₂ + ...</b>
  </div>

  <p>
    Here:
  </p>
  <ul>
    <li><b>β₀</b> is the constant/intercept</li>
    <li><b>β₁, β₂, ...</b> are the coefficients for the features</li>
  </ul>

  <p>
    The constant means the predicted value of <b>y</b> when all input values are 0.
  </p>

  <h3>Why do we add the Constant?</h3>
  <p>
    Without the constant, the model becomes:
  </p>

  <div class="box">
    <b>y = β₁x₁ + β₂x₂ + ...</b>
  </div>

  <p>
    This forces the regression line to pass through 0, which is often unrealistic.
    So we add the constant to allow the model to shift up or down and fit the data better.
  </p>

  <h3>How X Looks After add_constant</h3>
  <p>Before adding the constant:</p>
  <pre><code>R&D Spend | Administration | Marketing Spend | State_Florida | State_New York
----------------------------------------------------------------------------
165349.20 | 136897.80      | 471784.10       | 0              | 1
162597.70 | 151377.59      | 443898.53       | 0              | 0</code></pre>

  <p>After adding the constant:</p>
  <pre><code>const | R&D Spend | Administration | Marketing Spend | State_Florida | State_New York
--------------------------------------------------------------------------------------
1.0   | 165349.20 | 136897.80      | 471784.10       | 0              | 1
1.0   | 162597.70 | 151377.59      | 443898.53       | 0              | 0</code></pre>

  <p>
    The <b>const</b> column is all 1s because it lets the model learn the intercept value.
  </p>

  <h3>3. Fit the Model</h3>
  <pre><code>model = sm.OLS(y, X).fit()</code></pre>
  <p>
    This fits an Ordinary Least Squares (OLS) linear regression model using the current set of features.
  </p>

  <h3>4. Get the p-values</h3>
  <pre><code>p_values = model.pvalues.drop("const", errors="ignore")</code></pre>
  <p>
    This gets the p-value for each feature and ignores the constant.
  </p>

  <h2>What is a p-value?</h2>
  <p>
    A p-value helps measure whether a feature is statistically useful in the model.
  </p>

  <p>
    It answers this question:
    <b>"If this feature actually had no effect, how likely is it that we would still see a result like this by chance?"</b>
  </p>

  <table>
    <tr>
      <th>p-value</th>
      <th>Meaning</th>
    </tr>
    <tr>
      <td>Small (for example 0.01)</td>
      <td>Strong evidence that the feature matters</td>
    </tr>
    <tr>
      <td>Large (for example 0.70)</td>
      <td>The feature may not be useful</td>
    </tr>
  </table>

  <p>
    In backward elimination, features with large p-values are removed first.
  </p>

  <h2>What is the Significance Level?</h2>
  <pre><code>significance_level = 0.05</code></pre>
  <p>
    The significance level is the cutoff used to decide whether a feature stays or gets removed.
  </p>

  <table>
    <tr>
      <th>Condition</th>
      <th>Decision</th>
    </tr>
    <tr>
      <td>p-value ≤ 0.05</td>
      <td>Keep the feature</td>
    </tr>
    <tr>
      <td>p-value &gt; 0.05</td>
      <td>Remove the feature</td>
    </tr>
  </table>

  <p>
    We often use <b>0.05</b> because it is a common statistical threshold.
    It means we accept about a 5% chance of incorrectly thinking a feature is useful.
  </p>

  <h3>5. Check if there are any features left</h3>
  <pre><code>if p_values.empty:
    break</code></pre>
  <p>
    If no features remain, the loop stops.
  </p>

  <h3>6. Find the Largest p-value</h3>
  <pre><code>max_p_value = p_values.max()</code></pre>
  <p>
    This finds the least useful feature among the current features.
  </p>

  <h3>7. Remove the Worst Feature</h3>
  <pre><code>if max_p_value > significance_level:
    feature_to_remove = p_values.idxmax()
    print(f"Removing {feature_to_remove} with p-value {max_p_value:.4f}")
    X = X.drop(columns=[feature_to_remove])</code></pre>

  <p>
    If the largest p-value is above the significance level, that feature is removed.
  </p>

  <h3>8. Stop When All Features Are Significant</h3>
  <pre><code>else:
    break</code></pre>
  <p>
    If every feature has a p-value less than or equal to the significance level,
    the process stops.
  </p>

  <h3>9. Return the Final Data and Model</h3>
  <pre><code>return X, model</code></pre>
  <p>
    The function returns:
  </p>
  <ul>
    <li><b>X</b>: the final selected features</li>
    <li><b>model</b>: the final trained regression model</li>
  </ul>

  <h2>Simple Summary</h2>
  <div class="box">
    <p><b>Constant:</b> the intercept, or baseline prediction when all inputs are 0</p>
    <p><b>p-value:</b> tells us whether a feature is likely useful or not</p>
    <p><b>Significance level:</b> the threshold used to decide whether to keep or remove a feature</p>
    <p><b>Backward elimination:</b> repeatedly removes the least useful feature until only significant ones remain</p>
  </div>
